In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

prunings = [
'global-unstructured-L1',
   # 'local-unstructured-L1',
]
levels = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
target_class = 104

# ------------------------------------------------------------------
# 1. Collect data
# ------------------------------------------------------------------
data = []
for p in prunings:
   for l in levels:
      df = pd.read_csv(f'./neg/all-minmax-{p}-{l}.csv')
      temp = (
        df[(df['target_class_idx'] == target_class) & (df['order'] <= 4)]
        .sort_values('order')[['order', 'least_class_idx', 'least_class_count']]
        .reset_index(drop=True)
      )
      for _, row in temp.iterrows():
        data.append({
           "sparsity": float(l),
           "label":   int(row["least_class_idx"]),
           "count":   int(row["least_class_count"]),
        })

# ------------------------------------------------------------------
# 2. Build & clean DataFrame
# ------------------------------------------------------------------
df = pd.DataFrame(data)

df["sparsity"] = df["sparsity"].astype(float).round(2).astype(str)  # categorical hue
df["label"]   = df["label"].astype(int).astype(str)
df["count"]   = pd.to_numeric(df["count"], errors="coerce")
df = df.dropna(subset=["sparsity", "label", "count"])

# Collapse duplicate (label, sparsity) rows so each bar is unique
df = df.groupby(["sparsity", "label"], as_index=False)["count"].sum()

# ------------------------------------------------------------------
# 3. Consistent ordering
# ------------------------------------------------------------------
order    = sorted(df["label"].unique(),   key=int)    # numeric label order
hue_order = sorted(df["sparsity"].unique(), key=float)  # numeric sparsity order

# ------------------------------------------------------------------
# 4. Plot
# ------------------------------------------------------------------
plt.figure(figsize=(10, max(4, 0.5 * len(order))))
ax = sns.barplot(
   data=df,
   y="label",
   x="count",
   hue="sparsity",
   order=order,
   hue_order=hue_order,
   orient="h",
   estimator="sum",
   errorbar=None,
   palette="viridis",
)

# ------------------------------------------------------------------
# 5. Legend sanity check + styling
# ------------------------------------------------------------------
handles, labels = ax.get_legend_handles_labels()
plt.tight_layout()